In [0]:
!git clone -b sft_evaluation https://github.com/alapedriza1/mistral-7b-enterprise-function-calling.git

# 04 - Evaluation

Evaluates the fine-tuned adapter ([alapedriza/mistral-7b-function-calling-adapter](https://huggingface.co/alapedriza/mistral-7b-function-calling-adapter)) against the baseline Mistral-7B-Instruct-v0.3 on the 139-example test set.

Compares tool selection accuracy, JSON validity, argument matching, and no-tool restraint.

In [0]:
%pip install -q transformers accelerate bitsandbytes peft trl datasets tqdm

In [0]:
import sys
import os
import json
import pandas as pd
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

PROJECT_ROOT = "/kaggle/working/mistral-7b-enterprise-function-calling"
sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_jsonl
from src.inference import load_finetuned_model, run_inference_on_test_set
from src.evaluation import evaluate_results
from src.reporting import overall_summary, breakdown_by_category, breakdown_by_tool

In [0]:
DATA_DIR = f"{PROJECT_ROOT}/data"
RESULTS_DIR = f"{PROJECT_ROOT}/results"

test_data = load_jsonl(f"{DATA_DIR}/test.jsonl")
print(f"Test set: {len(test_data)} examples")

In [0]:
baseline_summary = pd.read_csv(f"{RESULTS_DIR}/baseline_summary.csv", index_col=0)
print(f"Loaded baseline summary from {RESULTS_DIR}/baseline_summary.csv")
baseline_summary

## Fine-Tuned Model

In [0]:
HF_REPO_ID = "alapedriza/mistral-7b-function-calling-adapter"
print(f"Loading fine-tuned model from {HF_REPO_ID}...", flush=True)
ft_model, ft_tokenizer = load_finetuned_model(HF_REPO_ID)
print("Running inference on test set...", flush=True)
finetuned_results = run_inference_on_test_set(ft_model, ft_tokenizer, test_data)

In [0]:
finetuned_eval_df = evaluate_results(finetuned_results)
finetuned_summary = overall_summary(finetuned_eval_df)
finetuned_summary

In [0]:
finetuned_cat = breakdown_by_category(finetuned_eval_df)
finetuned_cat

In [0]:
finetuned_tool = breakdown_by_tool(finetuned_eval_df, finetuned_results)
finetuned_tool

In [0]:
# Save fine-tuned results
with open(f"{RESULTS_DIR}/finetuned_results.jsonl", "w") as f:
    for r in finetuned_results:
        f.write(json.dumps(r) + "\n")

finetuned_eval_df.to_csv(f"{RESULTS_DIR}/finetuned_eval_df.csv", index=True)
finetuned_summary.to_csv(f"{RESULTS_DIR}/finetuned_summary.csv", index=True)
finetuned_cat.to_csv(f"{RESULTS_DIR}/finetuned_cat_breakdown.csv", index=True)
finetuned_tool.to_csv(f"{RESULTS_DIR}/finetuned_tool_breakdown.csv", index=True)

print(f"Results saved to {RESULTS_DIR}/")

In [0]:
# Failure-mode breakdown: non-overlapping buckets so counts sum to total errors.

tool_examples = finetuned_eval_df[finetuned_eval_df["category"] != "no_tool"]
n_tool = len(tool_examples)

# Build a waterfall: each bucket removes examples claimed by previous ones
parseable = tool_examples[tool_examples["json_valid"] == True]
correct_func = parseable[parseable["func_name_correct"] == True]

failure_buckets = {
    "JSON Parse Failures": len(tool_examples) - len(parseable),
    "Wrong Function Name": len(parseable) - len(correct_func),
    "Missing Required Fields": (correct_func["required_fields"] < 1.0).sum() if len(correct_func) else 0,
    "Hallucinated Params": (correct_func["hallucinated_params"] > 0).sum() if len(correct_func) else 0,
}

print(f"Tool-call examples: {n_tool}\n")
for label, count in failure_buckets.items():
    print(f"  {label:.<30} {count:>4} / {n_tool}  ({count/n_tool:.1%})")

# No-tool restraint (separate population)
no_tool = finetuned_eval_df[finetuned_eval_df["category"] == "no_tool"]
if len(no_tool) > 0:
    false_triggers = (no_tool["no_tool_restraint"] == False).sum()
    print(f"\nNo-Tool False Triggers: {false_triggers} / {len(no_tool)} "
          f"({false_triggers/len(no_tool):.1%})")

## Comparison

In [0]:
comparison = baseline_summary.rename(columns={"score": "Baseline"}).join(
    finetuned_summary.rename(columns={"score": "Fine-Tuned"})
)
comparison["Delta"] = comparison["Fine-Tuned"] - comparison["Baseline"]
comparison

In [0]:
baseline_cat = pd.read_csv(f"{RESULTS_DIR}/baseline_cat_breakdown.csv", index_col=0)

# Join on shared metrics, suffix to distinguish
cat_comparison = baseline_cat.drop(columns="n").add_suffix(" (Baseline)").join(
    finetuned_cat.drop(columns="n").add_suffix(" (Fine-Tuned)")
)

# Compute deltas for each metric
for metric in baseline_cat.columns.drop("n"):
    cat_comparison[f"{metric} (Delta)"] = cat_comparison[f"{metric} (Fine-Tuned)"] - cat_comparison[f"{metric} (Baseline)"]

# Reorder columns: group by metric [Baseline, Fine-Tuned, Delta]
ordered_cols = []
for metric in baseline_cat.columns.drop("n"):
    ordered_cols.extend([f"{metric} (Baseline)", f"{metric} (Fine-Tuned)", f"{metric} (Delta)"])

cat_comparison[ordered_cols]

## Key Observations

### Overall Performance (Fine-Tuned vs Baseline)

| Metric | Baseline | Fine-Tuned | Delta |
| --- | --- | --- | --- |
| JSON Validity Rate | 87.1% | 97.8% | +10.8 pp |
| Function Name Accuracy | 91.7% | 97.1% | +5.3 pp |
| Required Fields Completeness | 95.5% | 95.6% | +0.1 pp |
| Type Correctness | 100% | 100% | - |
| Enum Compliance | 100% | 99.3% | -0.7 pp |
| Argument Value Match | 76.8% | 83.7% | +6.8 pp |
| Avg Hallucinated Params | 0.008 | 0.000 | -0.008 |
| No-Tool Restraint | 100% | 100% | - |

### Failure Mode Breakdown (139 tool-call examples)

1. **JSON Parse Failures**: 2.2% (3/139), down from 12.9%. The dominant baseline failure mode is nearly eliminated.
2. **Wrong Function Name**: 2.9% (4/139), down from 7.2%. The model now rarely confuses semantically similar tools.
3. **Missing Required Fields**: 1.4% (2/139), down from 4.3%. Already rare at baseline, now marginal.
4. **Hallucinated Params**: 0.0% (0/139). Maintained - the model never invents parameters outside the schema.
5. **No-Tool False Triggers**: 0.0% (0/15). Perfect restraint preserved.

### Biggest Improvements by Category

| Category | JSON Valid (Delta) | Func Name (Delta) | Arg Match (Delta) | n |
| --- | --- | --- | --- | --- |
| multi_tool | +12.5 pp | +16.7 pp | +24.2 pp | 24 |
| complex | +16.2 pp | +7.0 pp | +12.4 pp | 37 |
| ambiguous | +18.8 pp | +1.4 pp | -1.0 pp | 16 |
| simple | +4.8 pp | +1.9 pp | +1.5 pp | 62 |

**multi_tool** saw the largest gains across all metrics - exactly the category where the baseline struggled most. Complex queries also improved substantially. Simple calls, already strong at baseline, saw modest refinement.

### Remaining Weaknesses

- **create_invoice** (62.5% JSON validity): Nested line-item arrays remain the hardest structural challenge, though argument match jumped to 96% when JSON is valid.
- **run_database_query** (71.4% function name, 51.2% argument match): Free-form SQL strings inside JSON and confusion with other tools persist.
- **send_notification** (71.4% required fields, 65.7% argument match): Multi-recipient structures still trip the model.
- **update_customer_record** (57.1% argument match): Complex nested update payloads remain difficult.

### What Fine-Tuning Fixed

The fine-tuning directly addressed the three weaknesses identified in the baseline evaluation:

1. **JSON formatting** - reduced parse failures from 12.9% to 2.2% (83% reduction). The model now reliably produces valid JSON even for complex schemas.
2. **Multi-tool call formatting** - JSON validity in multi_tool went from 75% to 87.5%, and argument match from 54.2% to 78.4%.
3. **Parameter hallucination** - eliminated entirely (0.008 to 0.000 avg hallucinated params).

### What Remains for Future Work

- **Argument value precision** (83.7%) is the main remaining gap - the model selects the right tool and produces valid JSON, but still misses exact parameter values ~16% of the time.
- **Enum compliance** dipped slightly (-0.7 pp) - one trade-off of the tool-trimming strategy where the model saw fewer full schemas during training.
- Tools with deeply nested objects (invoices, notifications, database queries) would benefit from additional targeted training examples.